# 10 - Scaling the Fraud-Path Calibration: A Negative Result Worth Keeping

Notebook 09 calibrated the temporal fraud gate against notebook 07's 218
near-duplicate clusters (6,150 narrative complaints) and landed on a clean
0% false-positive rate. That's one calibration set, drawn from one
relatively small slice of CFPB's data. This notebook scales the same
question up by roughly 34x -- not to "the full dataset" (the raw file
actually has 26.3 million rows, itself far bigger than anything tractable to
embed locally in one sitting) but to a much bigger, comparable sample -- and
asks whether the 0% result holds.

It doesn't. What follows is the chain of diagnosis that explains why, ending
in a genuinely different conclusion than "recalibrate the thresholds":
**temporal clustering, however it's measured, can't distinguish a popular
template going organically viral from a coordinated filing campaign.**
Both produce bursty, non-uniform timestamps. This isn't a threshold problem
-- four different statistical formulations of the same underlying test all
hit the same wall.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pickle

import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats

from src.retrieval import build_faiss_index


## The bigger sample

Drawn directly from `data/raw/complaints.csv` (26,339,914 rows -- `wc -l`
overcounts this because narrative text fields contain embedded newlines;
pandas' CSV parser gives the correct row count). Restricted to the same
2019-01-01 to 2022-12-31 window the existing `complaints_50k.parquet` was
drawn from, so this stays comparable to every prior notebook rather than
introducing a confound from CFPB's narrative-publication rate changing a lot
over time (it ranges from 0% pre-2015 to ~45% in 2017-2019 down to ~22% in
2025, partly real behavior change and partly a publication-lag artifact for
very recent complaints).

Sampling (~500k rows, ~90s) and embedding (207,874 narrative-bearing rows,
~67 minutes) were run as one-off background scripts rather than notebook
cells -- both are far too slow to re-run interactively every time this
notebook executes. Their outputs are loaded directly below.

In [2]:
df = pd.read_parquet("../data/processed/complaints_500k_narrative.parquet")
df["Date received"] = pd.to_datetime(df["Date received"])
with open("../data/processed/complaints_500k_narrative_embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date received'].min().date()} to {df['Date received'].max().date()}")
print(f"Embeddings: {embeddings.shape}")


Rows: 207,874
Date range: 2019-01-01 to 2022-12-31
Embeddings: (207874, 384)


## Why notebook 07's approach can't run here

Notebook 07 built a full N x N cosine similarity matrix via sklearn -- fine
at N=6,150 (~150MB). At N=207,874, that same dense matrix would be
~173GB. FAISS's `index.search()` avoids this: it computes the same exact
(not approximate) similarities internally but only returns the top-k per
query, so memory stays at N x k instead of N x N.

In [3]:
THRESHOLD = 0.85
K = 50
BATCH = 5000

index, embeddings_norm = build_faiss_index(embeddings)
print(f"Index built: {index.ntotal:,} vectors")

G = nx.Graph()
n = len(df)
G.add_nodes_from(range(n))

for batch_start in range(0, n, BATCH):
    batch_end = min(batch_start + BATCH, n)
    sims, idxs = index.search(embeddings_norm[batch_start:batch_end], K)
    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i
        for sim, j in zip(sims[local_i], idxs[local_i]):
            j = int(j)
            if j == -1 or j == global_i or sim < THRESHOLD:
                continue
            G.add_edge(global_i, j, weight=float(sim))

components = [c for c in nx.connected_components(G) if len(c) > 1]
components = sorted(components, key=len, reverse=True)

print(f"Edges: {G.number_of_edges():,}")
print(f"Duplicate clusters found (size > 1): {len(components):,}")
print(f"Cluster sizes (top 10): {[len(c) for c in components[:10]]}")
print(f"Total complaints involved: {sum(len(c) for c in components):,}")


Index built: 207,874 vectors


Edges: 1,803,230
Duplicate clusters found (size > 1): 9,635
Cluster sizes (top 10): [55192, 1334, 969, 394, 316, 293, 263, 241, 232, 223]
Total complaints involved: 92,613


## A 55,192-member "cluster" -- the first sign something's wrong

The biggest cluster contains over a quarter of the entire corpus. That's
not a template; notebook 07's biggest cluster, at 1/34th the data, was 353
members. Connected components are transitive -- if A~B and B~C, all three
land in one component even if A and C aren't directly similar at all. At
this density, the graph has enough edges for many genuinely distinct
templates to bridge into one blob via generic phrases common to several of
them. Check directly: do random pairs *within* this cluster actually look
similar to each other?

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

giant = list(components[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(giant, size=300, replace=False)
sim = cosine_similarity(embeddings[sample_idx])
np.fill_diagonal(sim, np.nan)

print(f"Giant cluster size: {len(giant):,}")
print(f"Random-pair similarity within it: mean={np.nanmean(sim):.3f}, median={np.nanmedian(sim):.3f}")
print(f"Fraction of random pairs >= 0.85: {(np.nan_to_num(sim, nan=0) >= 0.85).mean()*100:.1f}%")
print()
for i in rng.choice(giant, size=3, replace=False):
    print(f"--- row {i} ---")
    print(df.iloc[i]['Consumer complaint narrative'][:200])
    print()


Giant cluster size: 55,192
Random-pair similarity within it: mean=0.539, median=0.529
Fraction of random pairs >= 0.85: 3.5%

--- row 64083 ---
I've sent multiple letters requesting validation of my credit report. I have yet to receive anything i have asked for. Instead of providing me with the proof of verification which is a violation of FC

--- row 66771 ---
Fraudulent inquiries and inaccurate account information

--- row 85285 ---
In accordance with the fair credit Reporting act Account XXXX XXXX and Account XXXX have violated my rights. 
15 U.S.C 1681 section 602 A states i have the right to privacy.

15 U.S.C 1681 604 A. Sect



Confirmed: random pairs inside the "cluster" average 0.54 similarity, with
only ~3% actually clearing 0.85. It's a bridging artifact, not a template.

## Full coherence audit

Is this isolated to the one giant cluster, or pervasive? Sample up to 50
members from every cluster, compute mean pairwise similarity, and look at
the distribution before trusting any of this as a calibration set.

In [5]:
rows = []
for ci, cl in enumerate(components):
    cl = list(cl)
    n_sample = min(len(cl), 50)
    sample_idx = rng.choice(cl, size=n_sample, replace=False) if len(cl) > n_sample else np.array(cl)
    sim = cosine_similarity(embeddings[sample_idx])
    np.fill_diagonal(sim, np.nan)
    mean_sim = np.nanmean(sim) if n_sample > 1 else 1.0
    rows.append({"cluster_id": ci, "size": len(cl), "mean_sim": mean_sim})

audit = pd.DataFrame(rows)
print(audit["mean_sim"].describe())
print()
incoherent = audit[audit["mean_sim"] < 0.85]
print(f"Clusters below 0.85 mean similarity: {len(incoherent):,} / {len(audit):,}")
print(f"Complaints in them: {incoherent['size'].sum():,} / {audit['size'].sum():,}")


count    9635.000000
mean        0.943493
std         0.056991
min         0.564317
25%         0.894185
50%         0.960992
75%         0.999946
max         1.000000
Name: mean_sim, dtype: float64

Clusters below 0.85 mean similarity: 397 / 9,635
Complaints in them: 59,894 / 92,613


So: 396 out of 9,635 clusters (4.1%) are incoherent bridging artifacts, but
they account for 59,770 of 92,613 "duplicate" complaints (64.5%) -- almost
entirely the one giant cluster. The fix for tonight's purposes: drop
anything below the 0.85 mean-similarity bar. This is a coherence *filter*,
not a better clustering algorithm (Louvain modularity-based community
detection, which this project already used in notebooks 05/06, would likely
avoid this kind of chaining at the source -- worth a future revisit, not
done here).

In [6]:
clean_ids = set(audit[audit["mean_sim"] >= 0.85]["cluster_id"])
print(f"Clean clusters: {len(clean_ids):,}")
print(f"Complaints in clean clusters: {audit[audit['cluster_id'].isin(clean_ids)]['size'].sum():,}")


Clean clusters: 9,238
Complaints in clean clusters: 32,719


## Recalibrating notebook 09's gate against this much bigger, clean set

Same `temporal_suspicion_score` and the same three-condition combination
that produced 0% false positives on notebook 07's 113-cluster calibration
set (IQR-to-range ratio < 0.2, KS p-value < 0.05, total range <= 60 days).

In [7]:
def temporal_suspicion_score(dates):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)
    if len(dates) < 3:
        return None
    days = (dates - dates.min()).dt.days.astype(float).values
    total_range_days = float(days.max())
    if total_range_days == 0:
        return {"iqr_to_range_ratio": 0.0, "ks_pvalue": 0.0, "total_range_days": 0.0}
    q75, q25 = np.percentile(days, [75, 25])
    ratio = (q75 - q25) / total_range_days
    _, p = stats.kstest(days / total_range_days, "uniform")
    return {"iqr_to_range_ratio": ratio, "ks_pvalue": float(p), "total_range_days": total_range_days}


IQR_RATIO_THRESHOLD = 0.2
KS_PVALUE_THRESHOLD = 0.05
ABS_RANGE_THRESHOLD_DAYS = 60

calib_rows = []
for ci in clean_ids:
    cl = list(components[ci])
    s = temporal_suspicion_score(df.iloc[cl]["Date received"])
    if s is None:
        continue
    s["cluster_id"] = ci
    s["cluster_size"] = len(cl)
    calib_rows.append(s)

calibration_df = pd.DataFrame(calib_rows)
print(f"Scored clusters (>= 3 dates): {len(calibration_df):,}")

combined_flagged = calibration_df[
    (calibration_df["iqr_to_range_ratio"] < IQR_RATIO_THRESHOLD)
    & (calibration_df["ks_pvalue"] < KS_PVALUE_THRESHOLD)
    & (calibration_df["total_range_days"] <= ABS_RANGE_THRESHOLD_DAYS)
]
print(f"All three conditions: {len(combined_flagged):,} / {len(calibration_df):,} "
      f"({len(combined_flagged) / len(calibration_df) * 100:.2f}%)  "
      f"[notebook 09's smaller-scale result was 0 / 113 (0%)]")


Scored clusters (>= 3 dates): 2,053
All three conditions: 817 / 2,053 (39.80%)  [notebook 09's smaller-scale result was 0 / 113 (0%)]


**39.78%.** Complete breakdown of the result that calibrated cleanly at
1/34th the scale. Read a few of the flagged clusters directly before
concluding anything about why.

In [8]:
for ci in [105, 94]:
    cl = list(components[ci])
    sub = df.iloc[cl].sort_values("Date received")
    print(f"=== cluster {ci}, size {len(cl)} ===")
    print(f"Date range: {sub['Date received'].min().date()} to {sub['Date received'].max().date()}")
    print(f"Unique companies: {sub['Company'].nunique()}")
    print(sub["Company"].value_counts().head(3))
    print()
    for _, row in sub.head(2).iterrows():
        print(f"[{row['Date received'].date()}] {row['Company']}")
        print(row["Consumer complaint narrative"][:200])
        print("-" * 60)
    print()


=== cluster 105, size 30 ===
Date range: 2020-11-17 to 2020-11-24
Unique companies: 3
Company
EQUIFAX, INC.                             16
Experian Information Solutions Inc.       12
TRANSUNION INTERMEDIATE HOLDINGS, INC.     2
Name: count, dtype: int64

[2020-11-17] EQUIFAX, INC.
Upon reviewing my credit report, I noticed that there are several errors on there that are severely impacting my credit. Please investigate these accounts.
------------------------------------------------------------
[2020-11-17] Experian Information Solutions Inc.
Upon reviewing my credit report, I noticed that there are several errors on there that are severely impacting my credit. Please investigate these accounts.
------------------------------------------------------------

=== cluster 94, size 34 ===
Date range: 2021-09-07 to 2021-10-21
Unique companies: 28
Company
CAPITAL ONE FINANCIAL CORPORATION    3
PRESTIGE FINANCIAL SERVICES INC      2
I.C. System, Inc.                    2
Name: count, dtype: in

**Cluster 105** (30 members, 7-day window): the single most generic credit-
dispute boilerplate in the whole dataset ("Upon reviewing my credit report,
I noticed errors..."), hitting only the 3 credit bureaus. At 6,150 rows this
template had maybe a few dozen total uses spread across 4 years. At 207,874
rows it has thousands -- so *any* random week now contains dozens of
independent, unrelated uses, purely from volume.

**Cluster 94** (34 members, 28 different companies, 6-week window):
"*Please remove all accounts listed, my information was compromised in a
Data Breach*" -- independent people reacting to a real, dateable external
event. Genuinely non-uniform in time, for a completely mundane reason that
has nothing to do with coordination.

**The mechanism:** the gate's thresholds were implicitly tuned against the
cluster-size distribution of a 6,150-row corpus. At 34x the volume, popular
templates have enough raw occurrences that a short, dense window happens
constantly -- from sheer popularity, or a shared real-world trigger, neither
of which is fraud. This isn't a wrong-number problem. Try the
statistically rigorous version of the same test before concluding that.

## Attempt 1: a statistically proper version of the same test

The original test rescales each cluster's *own* dates to compare against a
uniform distribution -- which implicitly conditions on the observed range,
a circular comparison. The textbook-correct version: for `n` points drawn
uniformly at random over the *actual* full observation window
(here, 1,460 days), the range R follows a known distribution --
`R / T ~ Beta(n-1, 2)`. This gives a real p-value for "is this range
surprisingly small," properly accounting for both cluster size and the true
observation window, instead of an arbitrary absolute-day cutoff.

In [9]:
T_FULL_DAYS = (df["Date received"].max() - df["Date received"].min()).days
print(f"Full observation window: {T_FULL_DAYS} days")

def range_pvalue(dates, t_full_days):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)
    n = len(dates)
    if n < 3:
        return None
    days = (dates - df["Date received"].min()).dt.days.values
    r_observed = float(days.max() - days.min())
    return {"n": n, "range_pvalue": float(stats.beta.cdf(r_observed / t_full_days, n - 1, 2))}

rows = []
for ci in clean_ids:
    cl = list(components[ci])
    s = range_pvalue(df.iloc[cl]["Date received"], T_FULL_DAYS)
    if s is None:
        continue
    s["cluster_id"] = ci
    rows.append(s)

range_df = pd.DataFrame(rows)
flagged = (range_df["range_pvalue"] < 0.05).mean() * 100
print(f"Flagged (range_pvalue < 0.05): {flagged:.2f}%")

range_df["size_bucket"] = pd.cut(range_df["n"], bins=[3, 5, 10, 20, 50, 1500])
print(range_df.groupby("size_bucket", observed=True)["range_pvalue"].apply(lambda g: (g < 0.05).mean()))


Full observation window: 1460 days


Flagged (range_pvalue < 0.05): 88.36%
size_bucket
(3, 5]        0.862288
(5, 10]       0.903084
(10, 20]      0.940000
(20, 50]      0.949367
(50, 1500]    0.980392
Name: range_pvalue, dtype: float64


**88.36%, and worse: bigger clusters now get flagged *more* often** (86%
at size 3-5, up to 98% at size 50+) -- the exact opposite relationship from
before, and mathematically correct given the test. Real complaint filing is
never a literal homogeneous Poisson process over 4 years -- there's
seasonality, and template popularity rises and falls. Once a test's null
hypothesis is "this should look like flat random noise," any real dataset
eventually fails it as more data is added, whether or not anything
suspicious is happening. The math got more rigorous; the actual question
being asked was still wrong.

## Attempt 2: rescale against the corpus's own background rate

The standard fix for testing against a non-uniform null: estimate the
background event rate over time (here, overall complaint volume, which
captures real seasonality), then transform each date through that
background's cumulative distribution before testing for uniformity. This is
the time-rescaling theorem -- if it works, a genuinely innocent cluster's
rescaled positions should look uniform on [0, 1] even if the raw dates
don't, because the background curve already absorbs ordinary seasonal
variation.

In [10]:
all_dates_sorted = np.sort(df["Date received"].values.astype("datetime64[D]").astype(int))
N_BG = len(all_dates_sorted)

def rescale_to_background(dates):
    d = pd.to_datetime(pd.Series(dates)).values.astype("datetime64[D]").astype(int)
    return np.searchsorted(all_dates_sorted, d, side="right") / N_BG

def rescaled_test(dates):
    n = len(dates)
    if n < 3:
        return None
    tau = np.sort(rescale_to_background(dates))
    r_observed = tau[-1] - tau[0]
    range_p = stats.beta.cdf(r_observed, n - 1, 2)
    _, ks_p = stats.kstest(tau, "uniform")
    return {"n": n, "range_pvalue": range_p, "ks_pvalue_bg": ks_p}

rows = []
for ci in clean_ids:
    cl = list(components[ci])
    s = rescaled_test(df.iloc[cl]["Date received"])
    if s is None:
        continue
    s["cluster_id"] = ci
    rows.append(s)

rescaled_df = pd.DataFrame(rows)
print(f"Flagged (range_pvalue < 0.05): {(rescaled_df['range_pvalue'] < 0.05).mean()*100:.2f}%  [was 88.36% unrescaled]")
print(f"Flagged (ks_pvalue_bg < 0.05): {(rescaled_df['ks_pvalue_bg'] < 0.05).mean()*100:.2f}%")


Flagged (range_pvalue < 0.05): 87.97%  [was 88.36% unrescaled]
Flagged (ks_pvalue_bg < 0.05): 63.03%


## What did we find?

**87.97% and 63.05%.** Barely moved from the unrescaled version. The fix
didn't fail from a bug -- it failed because the actual mechanism is
different from what rescaling against *overall corpus volume* can capture.
Each template has its *own* adoption curve, and organic, entirely innocent
viral spread of a popular phrase is inherently bursty: it ramps up, peaks,
and fades, the same way any meme or script spreads through a population.
That burstiness is statistically indistinguishable from coordination using
shape-of-time-distribution alone, because **burstiness is a property of
popularity, not of coordination.** Four different formulations of this test
tonight -- naive thresholds (notebook 09), a parametric KS test, a
permutation-based KS test, a mathematically proper order-statistics test,
and a background-rate-rescaled version -- all hit the same wall once given
enough data for real statistical power.

**This is the actual finding, and it's bigger than a recalibration:**
temporal clustering, measured by the shape of a cluster's date distribution,
cannot on its own separate "this template went organically viral" from
"this was a coordinated filing campaign." Whatever actually distinguishes
them is probably not visible in this dataset at all -- shared submission
infrastructure, the same small number of real people behind many filings,
something CFPB's public complaint export doesn't carry. A more promising
direction, not built here, would normalize company-diversity by how many
companies could *plausibly* receive that complaint type (3 credit bureaus
isn't diverse, since those are the only valid recipients; 28 unrelated
companies for the same exact text is a different, genuinely wider net) --
but that's a different signal entirely, not a temporal one.

**What this means for notebook 09's `evaluate_fraud_signal()`:** its
three-condition temporal gate should be treated as **invalidated at scale**,
not just uncalibrated for this corpus size. The 0% result from notebook 09
was real but not informative about anything beyond the specific 113-cluster,
6,150-row sample it was measured on. `evaluate_fraud_signal()` remains
notebook-only (never promoted to `src/`), which in retrospect was the right
call -- there was nothing here ready to promote.

**Limitations:** the coherence-filtering step (drop clusters below 0.85
mean similarity) was a post-hoc patch on connected-components clustering,
not a fix to the clustering method itself -- Louvain community detection,
already used in notebooks 05/06, would likely avoid the bridging problem at
the source and is worth a future revisit. The background-rate rescaling
used overall corpus volume as the null, not anything template-specific --
a template-specific background would require an independent way to estimate
a template's "true" adoption rate that isn't circular with the cluster's
own member dates, which isn't solved here.